# Notebook 05 — Tableau Prep: Cleaning and RFM Features

This notebook turns the clean player table into BI-ready extracts for the Tableau
dashboards. It does three things:

1. Restricts the population to where churn lives (low and mid tiers).
2. Builds an **RFM** scoring model (Recency, Frequency, Engagement) and names a
   segment for every player.
3. Writes four tidy extracts that the dashboards read.

The scoring is computed once, here, so Tableau, Excel, and SQL all tell the same
story. The dashboards then read these files instead of recomputing anything.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path

DATA = Path("..")/"data"
OUT  = Path("..")/"tableau"; OUT.mkdir(exist_ok=True)

features = pd.read_csv(DATA/"player_features.csv")
sub = features[features.tier_band.isin(["low","mid"])].sort_values("puuid").reset_index(drop=True)
print(f"All engaged players: {len(features)}")
print(f"Low + mid (churn population): {len(sub)}  |  churned: {int(sub.churned.sum())}")

All engaged players: 704
Low + mid (churn population): 335  |  churned: 67


## 1. RFM scoring

RFM is a classic segmentation from retail, adapted here to a game:

- **Recency** = days since the last game, reversed (recent play scores higher).
- **Frequency** = total games in the window.
- **Engagement** = unique champions played (breadth), standing in for the usual
  Monetary axis since there is no spend data. Breadth is also a churn driver.

Each axis is scored 1 to 4 by **data quartile**, so the cut points come from the
players themselves rather than arbitrary thresholds.

In [2]:
R = [sub.days_since_last.quantile(q)  for q in (.25,.5,.75)]
F = [sub.total_games.quantile(q)       for q in (.25,.5,.75)]
E = [sub.unique_champions.quantile(q)  for q in (.25,.5,.75)]
print("Recency cutoffs   (days):", [round(x,1) for x in R])
print("Frequency cutoffs (games):", [round(x,1) for x in F])
print("Engagement cutoffs (champs):", [round(x,1) for x in E])

# Recency is reversed: fewer days since last game = higher score
sub["R_score"] = sub.days_since_last.map(lambda d: 4 if d<=R[0] else 3 if d<=R[1] else 2 if d<=R[2] else 1)
sub["F_score"] = sub.total_games.map(lambda g: 4 if g>=F[2] else 3 if g>=F[1] else 2 if g>=F[0] else 1)
sub["E_score"] = sub.unique_champions.map(lambda u: 4 if u>=E[2] else 3 if u>=E[1] else 2 if u>=E[0] else 1)
sub["RFM_total"] = sub.R_score + sub.F_score + sub.E_score
sub[["days_since_last","R_score","total_games","F_score","unique_champions","E_score","RFM_total"]].head()

Recency cutoffs   (days): [np.float64(1.0), np.float64(6.0), np.float64(21.5)]
Frequency cutoffs (games): [np.float64(52.0), np.float64(96.0), np.float64(157.5)]
Engagement cutoffs (champs): [np.float64(11.0), np.float64(20.0), np.float64(30.0)]


,days_since_last,R_score,total_games,F_score,unique_champions,E_score,RFM_total
0,10,2,64,2,25,3,7
1,21,2,196,4,32,4,10
2,17,2,42,1,14,2,5
3,5,3,37,1,13,2,6
4,0,4,117,3,11,2,9


## 2. Named segments

The Recency and Frequency scores combine into five action groups. Names beat
numbers because they map straight to a decision.

- **Champions** — recent and frequent. Safest.
- **Promising** — recent, lower frequency.
- **At risk** — frequent but gone quiet. The priority to re-engage.
- **Cooling** — moderate recency, slipping.
- **Dormant** — low on both. Highest churn.

In [3]:
def segment(r, f):
    if r >= 3 and f >= 3: return "Champions"
    if r >= 3:            return "Promising"
    if f >= 3 and r <= 2: return "At risk"
    if r == 2:            return "Cooling"
    return "Dormant"

sub["segment"]    = [segment(r, f) for r, f in zip(sub.R_score, sub.F_score)]
sub["player_id"]  = [f"PLAYER-{i+1:04d}" for i in range(len(sub))]
sub["churn_flag"] = sub.churned.astype(int)

seg_churn = (sub.groupby("segment")
               .agg(players=("churn_flag","size"), churn_rate=("churn_flag","mean"))
               .sort_values("churn_rate"))
seg_churn["churn_rate"] = (100*seg_churn.churn_rate).round(0)
print(seg_churn.to_string())

           players  churn_rate
segment                       
Champions      101         4.0
Promising       77         8.0
Cooling         36        28.0
At risk         69        32.0
Dormant         52        48.0


## 3. Extract 1 — players (one row per player)

The main extract. Features plus the RFM scores, the segment, and a clean
`player_id` to display instead of the long puuid.

In [4]:
players = sub[["player_id","tier","tier_band","primary_role","churn_flag","total_games",
                "days_since_last","unique_champions","games_per_active_day","win_rate",
                "avg_kda","games_trend","account_level","R_score","F_score","E_score",
                "RFM_total","segment"]].copy()
for col, n in [("win_rate",3),("games_per_active_day",2),("avg_kda",2)]:
    players[col] = players[col].round(n)
players.to_csv(OUT/"tableau_players.csv", index=False)
print("tableau_players.csv:", players.shape)
players.head(3)

tableau_players.csv: (335, 18)


,player_id,tier,tier_band,primary_role,churn_flag,total_games,days_since_last,unique_champions,games_per_active_day,win_rate,avg_kda,games_trend,account_level,R_score,F_score,E_score,RFM_total,segment
0,PLAYER-0001,SILVER,low,JUNGLE,0,64,10,25,2.37,0.422,2.22,-10,745,2,2,3,7,Cooling
1,PLAYER-0002,PLATINUM,mid,MIDDLE,1,196,21,32,5.60,0.526,3.38,180,733,2,4,4,10,At risk
2,PLAYER-0003,GOLD,mid,BOTTOM,0,42,17,14,2.21,0.524,2.98,42,553,2,1,2,5,Cooling


## 4. Extract 2 — drivers (churned vs retained)

Comparing eight behaviours across two groups, with the percent gap. Pre-shaped
to a tidy table so the dashboard plots one clean bar per metric, sidestepping the
mixed-scale problem (days vs win rate vs games).

In [5]:
metrics = [("Total games","total_games"),("Days since last","days_since_last"),
           ("Unique champions","unique_champions"),("Games per active day","games_per_active_day"),
           ("Win rate","win_rate"),("Avg KDA","avg_kda"),("Games trend","games_trend"),
           ("Account level","account_level")]
rows = []
for label, col in metrics:
    ret = sub.loc[sub.churn_flag==0, col].mean()
    chu = sub.loc[sub.churn_flag==1, col].mean()
    rows.append({"metric":label, "retained_avg":round(ret,2),
                 "churned_avg":round(chu,2), "diff_pct":round((chu-ret)/ret,3)})
drivers = pd.DataFrame(rows)
drivers.to_csv(OUT/"tableau_drivers.csv", index=False)
drivers

,metric,retained_avg,churned_avg,diff_pct
0,Total games,108.31,95.33,-0.120
1,Days since last,10.47,27.07,1.585
2,Unique champions,23.01,17.10,-0.257
3,Games per active day,3.58,4.28,0.196
4,Win rate,0.50,0.51,0.015
5,Avg KDA,3.22,3.65,0.135
6,Games trend,27.98,3.90,-0.861
7,Account level,440.08,380.54,-0.135


## 5. Extracts 3 and 4 — weekly activity and champion mix

Both come from the match-level table, restricted to the same low and mid players.

- **Weekly** powers the activity trend. The ISO **year** is corrected so late
  December and early January fall in the right week.
- **Champions** powers the champion-mix bars on the player detail page.

In [6]:
idmap = dict(zip(sub.puuid, sub.player_id))
cflag = dict(zip(sub.puuid, sub.churn_flag))
m = pd.read_csv(DATA/"player_match.csv")
m = m[m.puuid.isin(idmap)].copy()
m["player_id"]  = m.puuid.map(idmap)
m["churn_flag"] = m.puuid.map(cflag)

# champion mix per player
champ = (m.groupby(["player_id","champion"])
           .agg(games=("win","size"), wins=("win","sum")).reset_index())
champ["win_rate"] = (champ.wins/champ.games).round(3)
champ.to_csv(OUT/"tableau_player_champions.csv", index=False)

# weekly activity with ISO-year correction
m["gd"] = pd.to_datetime(m.game_date)
iso = m.gd.dt.isocalendar()
m["iso_year"], m["iso_week"] = iso.year.astype(int), iso.week.astype(int)
wk = (m.groupby(["iso_year","iso_week"])
        .agg(matches=("win","size"), active_players=("player_id","nunique")).reset_index())
ac = (m[m.churn_flag==1].groupby(["iso_year","iso_week"])
        .agg(active_churners=("player_id","nunique")).reset_index())
wk = wk.merge(ac, on=["iso_year","iso_week"], how="left").fillna({"active_churners":0})
wk["active_churners"] = wk.active_churners.astype(int)
wk["churner_share"]   = (wk.active_churners/wk.active_players).round(3)
wk = wk.sort_values(["iso_year","iso_week"]).reset_index(drop=True)
wk["week_label"] = wk.apply(lambda r: f"{r.iso_year}-W{int(r.iso_week):02d}", axis=1)
wk["week_index"] = range(1, len(wk)+1)
wk.to_csv(OUT/"tableau_weekly.csv", index=False)

print("tableau_player_champions.csv:", champ.shape)
print("tableau_weekly.csv:", wk.shape)
wk.head()

tableau_player_champions.csv: (7312, 5)
tableau_weekly.csv: (19, 8)


,iso_year,iso_week,matches,active_players,active_churners,churner_share,week_label,week_index
0,2025,47,63,29,8,0.276,2025.0-W47,1
1,2025,48,1350,147,31,0.211,2025.0-W48,2
2,2025,49,1504,161,32,0.199,2025.0-W49,3
3,2025,50,1211,154,26,0.169,2025.0-W50,4
4,2025,51,1083,138,21,0.152,2025.0-W51,5


## Summary

Four extracts, all derived from the same cleaned tables:

| File | Grain | Drives |
|---|---|---|
| `tableau_players.csv` | player (335) | KPIs, segments, RF heatmap, profile |
| `tableau_drivers.csv` | metric (8) | churned vs retained gap |
| `tableau_weekly.csv` | ISO week (19) | weekly activity trend |
| `tableau_player_champions.csv` | player-champion | champion mix |

The RFM scores and segments are computed once here, so every downstream tool
(Tableau, Excel, SQL) reads the same labels. That consistency is the point: one
analysis, many views.